In [47]:
# Set project root so "from src..." works. Edit the path below if you get ModuleNotFoundError.
import sys
from pathlib import Path
PROJECT_ROOT = Path("/Users/tomosborne/PycharmProjects/PowerPlan")  # <-- edit if your repo is elsewhere
if PROJECT_ROOT.exists():
    sys.path.insert(0, str(PROJECT_ROOT))

# Energy balancing model — Usage

1. **Energy options by tier (per kW)** — `get_energy_options(lat, lon)` returns solar and wind options at budget/mid/premium tiers. Optional: `start_date`, `end_date`.

2. **Monthly profiles** — `flux_df = get_flux(lat, lon)` then `get_monthly_profiles(flux_df, lat, lon)` gives a 12‑month solar/wind profile (kWh per kW per month) for capacity optimisation.

3. **Optimised system** — `get_optimised_system(lat, lon, annual_consumption_kwh=3500, pricing={...})` finds solar/wind capacity that minimises estimated cost. Use `annual_consumption_kwh` or `monthly_consumption_kwh=[...]` (12 values). Optional `pricing`: `solar_capex_per_kw`, `wind_capex_per_kw`, `grid_price_per_kwh`, `export_price_per_kwh`.

In [48]:
# Location and energy options by tier (run project-root cell first)
import importlib
import sys
import pandas as pd
# Force load from project (avoids stale cache or wrong package)
sys.modules.pop("src.models.energy_balancing", None)
importlib.invalidate_caches()
import src.models.energy_balancing as _eb
importlib.reload(_eb)
get_energy_options = _eb.get_energy_options
get_flux = _eb.get_flux
get_monthly_profiles = getattr(_eb, "get_monthly_profiles", None)
get_optimised_system = getattr(_eb, "get_optimised_system", None)
DEFAULT_PRICING = getattr(_eb, "DEFAULT_PRICING", {})
INSULATION_R_VALUE_SCALE = getattr(_eb, "INSULATION_R_VALUE_SCALE", 10.0)
if get_monthly_profiles is None:
    print("Warning: get_monthly_profiles not found in energy_balancing (update the module file).")

# Input: location by coordinates
latitude = 51.4552   # e.g. Bristol, UK
longitude = -2.5966

# Optional: restrict date range (default: next 7 days forecast)
# start_date, end_date = "2026-02-20", "2026-02-26"

options = get_energy_options(latitude, longitude)
options_df = pd.DataFrame(options)
options_df

,technology,tier,capacity_kw,annual_energy_kwh,cost_band,period_days,capacity_factor
0,solar,budget,1.0,366.1,low,7.0,4.18
1,solar,mid,1.0,411.9,medium,7.0,4.70
2,solar,premium,1.0,461.2,high,7.0,5.27
3,wind,budget,1.0,97.8,low,7.0,1.12
4,wind,mid,1.0,153.4,medium,7.0,1.75
5,wind,premium,1.0,212.3,high,7.0,2.42


In [49]:
# Summary: energy options at different price points (per kW capacity)
# Columns: technology, tier, annual_energy_kwh, cost_band, capacity_factor (%)
options_df[["technology", "tier", "annual_energy_kwh", "cost_band", "capacity_factor"]]

,technology,tier,annual_energy_kwh,cost_band,capacity_factor
0,solar,budget,366.1,low,4.18
1,solar,mid,411.9,medium,4.70
2,solar,premium,461.2,high,5.27
3,wind,budget,97.8,low,1.12
4,wind,mid,153.4,medium,1.75
5,wind,premium,212.3,high,2.42


In [50]:
# Monthly generation profiles (solar and wind, kWh per kW per month)
if get_monthly_profiles is None:
    print("get_monthly_profiles not available. Update src/models/energy_balancing.py.")
else:
    flux_df = get_flux(latitude, longitude)
    profiles = get_monthly_profiles(flux_df, latitude, longitude, solar_tier="mid", wind_tier="mid")
    print("Annual per kW: solar", round(profiles["annual_solar_per_kw"], 1), "kWh, wind", round(profiles["annual_wind_per_kw"], 1), "kWh")
    profiles["monthly_profile"]

Annual per kW: solar 411.9 kWh, wind 153.4 kWh


In [51]:
# Optimal system size and cost to meet your energy demand (capacity and cost minimised)
if get_optimised_system is None:
    print("get_optimised_system not available. Update src/models/energy_balancing.py.")
else:
    annual_consumption_kwh = 3500  # your energy demand (kWh/year)
    years = 25.0  # horizon for total cost and payback

    # --- Capacity inputs (solar panels and turbines) ---
    # Option A: Let the optimiser search for best size in this range:
    solar_max_kw = 20.0   # max solar capacity to consider (kWp)
    wind_max_kw = 10.0   # max wind capacity to consider (kW)
    step_kw = 0.5        # step size for search (e.g. 0.5 = 0, 0.5, 1.0, ... kW)
    # Option B: Fix a specific size (set both to evaluate that system; set to None to use Option A)
    fixed_solar_kw = None  # e.g. 4.0 for 4 kWp solar
    fixed_wind_kw = None   # e.g. 2.0 for 2 kW wind turbine

    min_met_pct = 50.0    # require this % of demand from solar/wind (0 = cost-only; 50 or 100 = meet demand)

    # --- Insulation: R-value in m²·K/W (reduces heating share of demand before sizing) ---
    insulation_r_value = 0.0   # 0 = none; typical 1–6 m²·K/W; higher = better
    heating_fraction = 0.6    # share of demand that is space heating (e.g. 0.6 = 60%)

    result = get_optimised_system(
        latitude, longitude,
        annual_consumption_kwh=annual_consumption_kwh,
        min_demand_met_from_gen_pct=min_met_pct,
        years=years,
        solar_max_kw=solar_max_kw,
        wind_max_kw=wind_max_kw,
        step_kw=step_kw,
        fixed_solar_kw=fixed_solar_kw,
        fixed_wind_kw=fixed_wind_kw,
        insulation_r_value=insulation_r_value,
        heating_fraction=heating_fraction,
    )
    opt = result["optimisation"]
    print("--- Optimal system to meet your energy demand ---")
    if result.get("insulation_r_value", 0) > 0:
        print("Insulation: R =", result["insulation_r_value"], "m²·K/W (heating share", int(result.get("heating_fraction", 0.6) * 100), "%)")
        print("Demand: before insulation", result.get("annual_demand_before_insulation_kwh"), "kWh → after", result.get("annual_demand_after_insulation_kwh"), "kWh")
    print("Demand (annual, used for sizing):", opt.get("annual_demand_kwh", annual_consumption_kwh), "kWh")
    print("Capacity: solar", opt["optimal_solar_kw"], "kWp, wind", opt["optimal_wind_kw"], "kW  (total", opt.get("total_capacity_kw", opt["optimal_solar_kw"] + opt["optimal_wind_kw"]), "kW)")
    print("Cost: capex £" + str(opt["capex"]) + ", grid (1 yr) £" + str(opt["annual_opex_import_cost"]) + ", export £" + str(opt["annual_export_revenue"]))
    print("Total cost over", int(opt.get("years_considered", years)), "years: £", opt.get("cost_over_years", opt["total_annual_cost_estimate"]))
    print("Payback (solar/wind vs grid-only):", str(opt.get("payback_years") or "—") + " years")
    print("Demand met from own generation:", str(opt.get("demand_met_from_generation_pct", "—")) + "%")
    if opt["optimal_solar_kw"] == 0 and opt["optimal_wind_kw"] == 0 and min_met_pct == 0:
        print("(All zeros: at default prices, grid-only is cheapest. Set min_met_pct=50 or 100 to size solar/wind.)")
    print("-----------------------------------------------")
    opt["monthly_balance"]

--- Optimal system to meet your energy demand ---
Demand (annual, used for sizing): 3500.0 kWh
Capacity: solar 6.5 kWp, wind 0.0 kW  (total 6.5 kW)
Cost: capex £9750.0, grid (1 yr) £348.35, export £1.77
Total cost over 25 years: £ 18414.48
Payback (solar/wind vs grid-only): 18.5 years
Demand met from own generation: 61.2%
-----------------------------------------------
